In [2]:
from pickletools import optimize

import torch
print(torch.cuda.is_available())
from sympy import true
from torch import nn
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.data import  Dataset
import torch.nn.functional as F

True


In [3]:
import torch.nn.functional as F

df = pd.read_csv("./data/rps.csv")


class RPSDataset(Dataset):
    def __init__(self, sequence_length):
        self.csv_path = "./data/rps.csv"
        self.sequence_length = sequence_length
        x = df.iloc[:,[-4,-3]].values
        moves = torch.tensor(x, dtype=torch.long)                       # (N, 2)
        self.data = F.one_hot(moves, num_classes=3).reshape(len(moves), -1).float()  # (N, 6)
        y = df.iloc[:,-3].values
        self.target = torch.tensor(y, dtype=torch.long)                 # stays 0/1/2
    def __len__(self):
        return len(self.data) - self.sequence_length
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.sequence_length]
        y = self.target[idx + self.sequence_length]
        return x, y

dataset = RPSDataset(sequence_length=5)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [5]:
class RPSNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 3)
    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        return self.fc(out[:,-1])

model = RPSNN(input_size=6,hidden_size=64,num_layers=2).to(device)


In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
def train(dataloader, model, loss_fn, optimizer):
    size = len(train_loader)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")





In [ ]:
epochs = 50
print(len(dataset))
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_loader, model, loss_fn, optimizer)
print("Done!")
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

97892
Epoch 1
-------------------------------
loss: 1.107936  [   64/ 1530]
loss: 1.098686  [ 6464/ 1530]
loss: 1.100015  [12864/ 1530]
loss: 1.093275  [19264/ 1530]
loss: 1.100811  [25664/ 1530]
loss: 1.097349  [32064/ 1530]
loss: 1.099615  [38464/ 1530]
loss: 1.103287  [44864/ 1530]
loss: 1.095772  [51264/ 1530]
loss: 1.099517  [57664/ 1530]
loss: 1.096673  [64064/ 1530]
loss: 1.098582  [70464/ 1530]
loss: 1.098967  [76864/ 1530]
loss: 1.103758  [83264/ 1530]
loss: 1.101333  [89664/ 1530]
loss: 1.100555  [96064/ 1530]
Epoch 2
-------------------------------
loss: 1.099966  [   64/ 1530]
loss: 1.093972  [ 6464/ 1530]
loss: 1.097574  [12864/ 1530]
loss: 1.099766  [19264/ 1530]
loss: 1.097052  [25664/ 1530]
loss: 1.094609  [32064/ 1530]
loss: 1.100246  [38464/ 1530]
loss: 1.097817  [44864/ 1530]
loss: 1.096163  [51264/ 1530]
loss: 1.096105  [57664/ 1530]
loss: 1.093643  [64064/ 1530]
loss: 1.098361  [70464/ 1530]
loss: 1.099751  [76864/ 1530]
loss: 1.098856  [83264/ 1530]
loss: 1.098252